In [4]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

#Data Path
DATA_DIR = Path.home() / "Desktop" / "MathMethodsFinalSleepData"
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SLEEP_STAGES = ["Sleep stage W", "Sleep stage N", "Sleep stage R"]

FEMALE_SUBJECTS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 20, 21, 22, 23, 24]
MALE_SUBJECTS   = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 30, 31, 32, 33, 34]

def find_hypnogram_file(subject_num: int) -> Path | None:
    pattern = f"SC4{subject_num:02d}1*-Hypnogram.edf"
    matches = list(DATA_DIR.glob(pattern))

    if len(matches) == 0:
        print(f"[MISSING] No file found for subject {subject_num}")
        return None
    if len(matches) > 1:
        print(f"[WARN] Multiple matches for subject {subject_num}: {[m.name for m in matches]}")
        return matches[0]

    return matches[0]

def build_sleep_state_seq(annotations):
    stage_sequence = []
    for i in range(len(annotations)):
        num_epochs = int(annotations.duration[i] / 30)
        for j in range(num_epochs):
            desc = annotations.description[i]
            if "1" in desc or "2" in desc or "3" in desc or "4" in desc:
                stage_sequence.append("Sleep stage N")
            else:
                stage_sequence.append(annotations.description[i])
    stage_sequence = np.array(stage_sequence)
    return stage_sequence

#Split into late and early night
def build_early_late_seqs(stage_sequence):
    total_dur_sec   = len(stage_sequence) * 30
    total_dur_hours = int(total_dur_sec / 3600)
    stages_per_hour = int(len(stage_sequence) / total_dur_hours)

    #Early night is 8-12 and Late Night is 12-16
    early_night_seq = stage_sequence[8  * stages_per_hour : 12 * stages_per_hour]
    late_night_seq  = stage_sequence[12 * stages_per_hour : 16 * stages_per_hour]

    return early_night_seq, late_night_seq

def build_transition_matrix(sequence, sleep_stages):
    sequence = np.delete(sequence, np.where(sequence == "Sleep stage ?"))
    sequence = np.delete(sequence, np.where(sequence == "Movement time"))

    stage_sequence_numbered = []
    for stage in sequence:
        stage_sequence_numbered.append(sleep_stages.index(stage))

    stage_count_matrix = np.zeros((3, 3))
    for n in np.arange(len(stage_sequence_numbered) - 1):
        stage_count_matrix[stage_sequence_numbered[n], stage_sequence_numbered[n + 1]] += 1

    transition_matrix = np.zeros((3, 3))
    for i in np.arange(3):
        trans_to_state_i = np.sum(stage_count_matrix[i])
        for j in np.arange(3):
            if trans_to_state_i > 0:
                transition_matrix[i, j] = stage_count_matrix[i, j] / trans_to_state_i
            else:
                transition_matrix[i, j] = 0

    return transition_matrix

def subject_transition_matrix(annotations, sleep_stages):
    all_states = build_sleep_state_seq(annotations)
    early, late = build_early_late_seqs(all_states)
    early_night_matrix = build_transition_matrix(early, sleep_stages)
    late_night_matrix  = build_transition_matrix(late, sleep_stages)
    return early_night_matrix, late_night_matrix

def average_matrices(matrix_list):
    stacked = np.array(matrix_list)
    avg = stacked.mean(axis=0)
    
    #re-normalize each row to sum to 1 (some subjects have zero transitions from rare states, which pulls the averaged row below 1)
    row_sums = avg.sum(axis=1, keepdims=True)
    avg = np.divide(avg, row_sums, out=np.zeros_like(avg), where=row_sums > 0)
    
    return avg

#Heatmap
def plot_heatmap(matrix, title, save_path=None):
    labels = ["W", "N", "R"]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, vmin=0, vmax=1)

    ax.set_xticks(range(3)); ax.set_xticklabels(labels)
    ax.set_yticks(range(3)); ax.set_yticklabels(labels)

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color="black")

    ax.set_xlabel("Next state")
    ax.set_ylabel("Current state")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

def process_group(subject_list, group_name):
    early_matrices = []
    late_matrices  = []

    for subj in subject_list:
        fpath = find_hypnogram_file(subj)
        if fpath is None:
            continue

        annotations = mne.read_annotations(str(fpath))
        early_mat, late_mat = subject_transition_matrix(annotations, SLEEP_STAGES)
        early_matrices.append(early_mat)
        late_matrices.append(late_mat)

    n = len(early_matrices)

    if n == 0:
        return None, None, [], []

    avg_early = average_matrices(early_matrices)
    avg_late  = average_matrices(late_matrices)

    return avg_early, avg_late, early_matrices, late_matrices

for group_name, subject_list in [("Female", FEMALE_SUBJECTS), ("Male", MALE_SUBJECTS)]:
    avg_early, avg_late, early_mats, late_mats = process_group(subject_list, group_name)

    if avg_early is None:
        continue
    
    if group_name == "Female":
        early_matrices_f, late_matrices_f = early_mats, late_mats
        female_early, female_late = avg_early, avg_late

    else:
        early_matrices_m, late_matrices_m = early_mats, late_mats
        male_early, male_late = avg_early, avg_late

    np.set_printoptions(precision=3, suppress=True)
    print(f"{group_name} - Average EARLY transition matrix (hrs 8-12):")
    print(avg_early)
    print(f"\n{group_name} - Average LATE transition matrix (hrs 12-16):")
    print(avg_late)

    plot_heatmap(
        avg_early,
        title=f"{group_name} - Avg Early Night (hrs 8-12, n={len(subject_list)})",
        save_path=OUTPUT_DIR / f"{group_name.lower()}_avg_early_heatmap.png",
    )
    plot_heatmap(
        avg_late,
        title=f"{group_name} - Avg Late Night (hrs 12-16, n={len(subject_list)})",
        save_path=OUTPUT_DIR / f"{group_name.lower()}_avg_late_heatmap.png",
    )

[MISSING] No file found for subject 0
[MISSING] No file found for subject 1
[MISSING] No file found for subject 2
[MISSING] No file found for subject 3
[MISSING] No file found for subject 4
[MISSING] No file found for subject 5
[MISSING] No file found for subject 6
[MISSING] No file found for subject 7
[MISSING] No file found for subject 8
[MISSING] No file found for subject 9
[MISSING] No file found for subject 20
[MISSING] No file found for subject 21
[MISSING] No file found for subject 22
[MISSING] No file found for subject 23
[MISSING] No file found for subject 24
[MISSING] No file found for subject 10
[MISSING] No file found for subject 11
[MISSING] No file found for subject 12
[MISSING] No file found for subject 13
[MISSING] No file found for subject 14
[MISSING] No file found for subject 15
[MISSING] No file found for subject 16
[MISSING] No file found for subject 17
[MISSING] No file found for subject 18
[MISSING] No file found for subject 19
[MISSING] No file found for subject

In [5]:
#Hypnogram Plot

def plot_hypnogram(subject_num, sex_label):
    fpath = find_hypnogram_file(subject_num)
    if fpath is None:
        print(f"File not found for subject {subject_num}")
        return

    ann = mne.read_annotations(str(fpath))

    #build 30-second sequence with times
    stages, times = [], []
    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        if "1" in desc or "2" in desc or "3" in desc or "4" in desc:
            state = "N"
        elif "W" in desc:
            state = "W"
        elif desc == "Sleep stage R":
            state = "R"
        else:
            continue
        n_epochs = int(round(duration / 30.0))
        for k in range(n_epochs):
            stages.append(state)
            times.append((onset + 30.0 * k) / 3600)  #convert to hours

    #map stages to y-axis values (Wake=2, REM=1, NonREM=0)
    stage_to_y = {"W": 2, "R": 1, "N": 0}
    y_vals = [stage_to_y[s] for s in stages]

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.step(times, y_vals, where="post", color="steelblue", linewidth=0.8)
    ax.fill_between(times, y_vals, 2, step="post", alpha=0.3, color="steelblue")

    #shade early and late windows
    ax.axvspan(8,  12, alpha=0.1, color="green",  label="Early window (hrs 8-12)")
    ax.axvspan(12, 16, alpha=0.1, color="orange", label="Late window (hrs 12-16)")

    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["Non-REM", "REM", "Wake"])
    ax.set_xlabel("Hours from recording start")
    ax.set_title(f"Hypnogram — Subject {subject_num} ({sex_label})")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"hypnogram_subj{subject_num:02d}.png", dpi=300, bbox_inches="tight")
    plt.show()

#Plot one female and one male subject as examples
plot_hypnogram(0, "Female")
plot_hypnogram(10, "Male")

[MISSING] No file found for subject 0
File not found for subject 0
[MISSING] No file found for subject 10
File not found for subject 10


In [6]:
#Boostrap Analysis

N_BOOTSTRAP = 1000
ALPHA = 0.05
rng = np.random.default_rng(42)

#Resampling subjects with replacement and average
def bootstrap_avg_matrix(matrices):
    matrices = np.array(matrices)
    n = len(matrices)
    boot_means = np.zeros((N_BOOTSTRAP, 3, 3))
    for b in range(N_BOOTSTRAP):
        sample_idx = rng.integers(0, n, size=n)   #random resample with replacement
        boot_means[b] = matrices[sample_idx].mean(axis=0)
    return boot_means

#Confidence Intervals
def print_bootstrap_ci(avg_mat, boot_means, group_name, window_label):
    lo = np.percentile(boot_means, 2.5,  axis=0)
    hi = np.percentile(boot_means, 97.5, axis=0)
    print(f"\n{group_name} | {window_label} — 95% Bootstrap CI")
    print("-" * 45)
    for i, from_s in enumerate(["W", "N", "R"]):
        for j, to_s in enumerate(["W", "N", "R"]):
            print(f"  P({from_s}->{to_s}): "
                  f"mean={avg_mat[i,j]:.3f}  "
                  f"95% CI [{lo[i,j]:.3f}, {hi[i,j]:.3f}]")
    return lo, hi

#run for each group and window
boot_f_early = bootstrap_avg_matrix(early_matrices_f)
boot_f_late  = bootstrap_avg_matrix(late_matrices_f)
boot_m_early = bootstrap_avg_matrix(early_matrices_m)
boot_m_late  = bootstrap_avg_matrix(late_matrices_m)

lo_fe, hi_fe = print_bootstrap_ci(female_early, boot_f_early, "Female", "Early (hrs 8-12)")
lo_fl, hi_fl = print_bootstrap_ci(female_late,  boot_f_late,  "Female", "Late (hrs 12-16)")
lo_me, hi_me = print_bootstrap_ci(male_early,   boot_m_early, "Male",   "Early (hrs 8-12)")
lo_ml, hi_ml = print_bootstrap_ci(male_late,    boot_m_late,  "Male",   "Late (hrs 12-16)")

#Permutation Test — Early vs. Late (stationarity)
#Null hypothesis: transition probabilities do not differ between early and late night (i.e. the Markov chain is stationary).
#For each permutation, randomly swap early/late labels within each subject and recompute the group difference. p-value = fraction of permutations >= observed difference.

def permutation_test_early_vs_late(early_mats, late_mats, group_name):
    early_mats = np.array(early_mats)
    late_mats  = np.array(late_mats)
    n = len(early_mats)

    #observed mean absolute difference between avg early and avg late
    obs_diff = np.abs(early_mats.mean(axis=0) - late_mats.mean(axis=0)).mean()

    null_diffs = np.zeros(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        #randomly flip early and late for each subject independently
        swap = rng.integers(0, 2, size=n).astype(bool)
        perm_early = np.where(swap[:, None, None], late_mats,  early_mats)
        perm_late  = np.where(swap[:, None, None], early_mats, late_mats)
        null_diffs[b] = np.abs(perm_early.mean(axis=0) - perm_late.mean(axis=0)).mean()

    p_value = (null_diffs >= obs_diff).mean()
    conclusion = "REJECT null (non-stationary)" if p_value < ALPHA else "FAIL TO REJECT null (stationary)"
    print(f"\n{group_name} — Early vs. Late Permutation Test")
    print(f"  Observed difference = {obs_diff:.4f}")
    print(f"  p-value = {p_value:.4f}  →  {conclusion}")

    #plot null distribution with observed statistic marked
    plt.figure(figsize=(7, 4))
    plt.hist(null_diffs, bins=40, color="steelblue", alpha=0.7, label="Null distribution")
    plt.axvline(obs_diff, color="red", linewidth=2, label=f"Observed = {obs_diff:.4f}")
    plt.hist(null_diffs[null_diffs >= obs_diff], bins=40, color="red", alpha=0.4, label="p-value region")
    plt.xlabel("Mean absolute difference in transition probabilities")
    plt.ylabel("Count")
    plt.title(f"{group_name}: Early vs. Late\np-value = {p_value:.4f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{group_name.lower()}_perm_early_vs_late.png", dpi=300, bbox_inches="tight")
    plt.show()

    return p_value

permutation_test_early_vs_late(early_matrices_f, late_matrices_f, "Female")
permutation_test_early_vs_late(early_matrices_m, late_matrices_m, "Male")

#Permutation Test — Female vs. Male
#Null hypothesis: transition probabilities do NOT differ between male and female subjects.
#Shuffle sex labels across all subjects and recompute the male vs. female difference under the null.

def permutation_test_sex(early_mats_f, early_mats_m, late_mats_f, late_mats_m):
    for window_label, mats_f, mats_m in [
        ("Early (hrs 8-12)", early_mats_f, early_mats_m),
        ("Late (hrs 12-16)", late_mats_f,  late_mats_m),
    ]:
        mats_f = np.array(mats_f)
        mats_m = np.array(mats_m)
        all_mats = np.concatenate([mats_f, mats_m], axis=0)
        n_f, n_total = len(mats_f), len(all_mats)

        #observed difference between female and male averages
        obs_diff = np.abs(mats_f.mean(axis=0) - mats_m.mean(axis=0)).mean()

        null_diffs = np.zeros(N_BOOTSTRAP)
        for b in range(N_BOOTSTRAP):
            #shuffle sex labels across all subjects
            perm_idx = rng.permutation(n_total)
            perm_f   = all_mats[perm_idx[:n_f]]
            perm_m   = all_mats[perm_idx[n_f:]]
            null_diffs[b] = np.abs(perm_f.mean(axis=0) - perm_m.mean(axis=0)).mean()

        p_value = (null_diffs >= obs_diff).mean()
        conclusion = "REJECT null (sex difference exists)" if p_value < ALPHA else "FAIL TO REJECT null (no sex difference)"
        print(f"\nFemale vs. Male | {window_label}")
        print(f"  Observed difference = {obs_diff:.4f}")
        print(f"  p-value = {p_value:.4f}  →  {conclusion}")

        plt.figure(figsize=(7, 4))
        plt.hist(null_diffs, bins=40, color="steelblue", alpha=0.7, label="Null distribution")
        plt.axvline(obs_diff, color="red", linewidth=2, label=f"Observed = {obs_diff:.4f}")
        plt.hist(null_diffs[null_diffs >= obs_diff], bins=40, color="red", alpha=0.4, label="p-value region")
        plt.xlabel("Mean absolute difference in transition probabilities")
        plt.ylabel("Count")
        plt.title(f"Female vs. Male | {window_label}\np-value = {p_value:.4f}")
        plt.legend()
        plt.tight_layout()
        fname = window_label.split()[0].lower()
        plt.savefig(OUTPUT_DIR / f"sex_perm_{fname}.png", dpi=300, bbox_inches="tight")
        plt.show()

permutation_test_sex(early_matrices_f, early_matrices_m, late_matrices_f, late_matrices_m)

NameError: name 'early_matrices_f' is not defined

In [ ]:
#DWELL TIME & STATIONARY DISTRIBUTION
#dwell time = avg number of steps spent in state i for females/males early/late in the night

def dwell_times(trans_matrix):
    diagonals = np.diag(trans_matrix)
    dwell = np.zeros(3)
    for i in range(len(diagonals)):
        dwell[i] = 1 / (1 - diagonals[i])
    return dwell

def stationary_dist(trans_matrix):
    evals, evecs = np.linalg.eig(trans_matrix.T)
    idx = np.argmin(np.abs(evals - 1))
    pi = np.real(evecs[:, idx]) #take real part
    pi = pi / np.sum(pi) #normalize so probabilities sum to 1
    return pi

labels = ["W", "N", "R"]

for group_name, mat_early, mat_late in [
    ("Female", female_early, female_late),
    ("Male",   male_early,   male_late),
]:
    print(f"{group_name} — Dwell Times & Stationary Distribution")

    for window_label, mat in [("Early (hrs 8-12)", mat_early),
                               ("Late  (hrs 12-16)", mat_late)]:
        dwell  = dwell_times(mat)
        st_dist = stationary_dist(mat)

        print(f"\n  {window_label}")
        print(f"  {'State':<8} {'Dwell (epochs)':<18} {'Dwell (mins)':<16} {'Stationary dist'}")
        print(f"  {'-'*58}")
        for i, s in enumerate(labels):
            print(f"  {s:<8} {dwell[i]:<18.2f} {dwell[i]*5:<16.2f} {st_dist[i]:.3f}")

: 

https://ericmjl.github.io/essays-on-data-science/machine-learning/markov-models/ - Used to Generate code for second-order Markov Chain

In [ ]:
#Second-Order Markov Chain — built using the above website along with claude

from collections import defaultdict

STAGE_LABELS = ["W", "N", "R"]

def build_second_order_matrix(sequence):
    # count transitions
    counts = defaultdict(lambda: np.zeros(3))
    for t in range(len(sequence) - 2):
        pair = (sequence[t], sequence[t + 1])
        next_state = sequence[t + 2]
        counts[pair][next_state] += 1

    # normalize each row
    matrix = {}
    for pair, row in counts.items():
        total = row.sum()
        matrix[pair] = row / total if total > 0 else row

    return matrix

def average_second_order_matrices(seq_list):
    all_counts = defaultdict(lambda: np.zeros(3))

    # pool raw counts across all subjects then normalize
    for seq in seq_list:
        for t in range(len(seq) - 2):
            pair = (seq[t], seq[t + 1])
            all_counts[pair][seq[t + 2]] += 1

    matrix = {}
    for pair, row in all_counts.items():
        total = row.sum()
        matrix[pair] = row / total if total > 0 else row

    return matrix

def print_second_order_scratch(matrix, title):
    print(f"\n{title}")
    print(f"  {'From (prev,curr)':<20} {'->W':>8} {'->N':>8} {'->R':>8}")
    print(f"  {'-'*46}")
    for (prev, curr) in [(i,j) for i in range(3) for j in range(3)]:
        if (prev, curr) in matrix:
            row = matrix[(prev, curr)]
            p = STAGE_LABELS[prev]
            c = STAGE_LABELS[curr]
            print(f"  ({p},{c}){'':>14}"
                  f"{row[0]:>8.3f}{row[1]:>8.3f}{row[2]:>8.3f}"
                  f"  sum={row.sum():.3f}")

def log_likelihood_second_order_scratch(seq, matrix):
    ll = 0.0
    for t in range(len(seq) - 2):
        pair = (seq[t], seq[t + 1])
        if pair in matrix:
            prob = matrix[pair][seq[t + 2]]
            if prob > 0:
                ll += np.log(prob)
    return ll

#Run for both groups
print("SECOND-ORDER MARKOV")

for group_name, early_seqs, late_seqs, mat_early_1st, mat_late_1st in [
    ("Female", f_early_seqs, f_late_seqs, female_early, female_late),
    ("Male",   m_early_seqs, m_late_seqs, male_early,   male_late),
]:
    print(f"\n{'='*55}\n{group_name}\n{'='*55}")

    #fit pooled second-order matrices
    mat2_early = average_second_order_matrices(early_seqs)
    mat2_late  = average_second_order_matrices(late_seqs)

    print_second_order_scratch(mat2_early, f"{group_name} Early (hrs 8-12)")
    print_second_order_scratch(mat2_late,  f"{group_name} Late (hrs 12-16)")

    #AIC/BIC comparison
    print(f"\n{group_name} — AIC/BIC (scratch 2nd order vs 1st order)")
    print(f"  {'Subj':<6} {'Window':<8} {'AIC_1st':>9} {'AIC_2nd':>9} {'BIC_1st':>9} {'BIC_2nd':>9} {'Winner':>8}")
    print(f"  {'-'*60}")

    subj_list  = FEMALE_SUBJECTS if group_name == "Female" else MALE_SUBJECTS
    totals = {"a1e":0,"a2e":0,"b1e":0,"b2e":0,"a1l":0,"a2l":0,"b1l":0,"b2l":0}

    for idx, (e_seq, l_seq) in enumerate(zip(early_seqs, late_seqs)):
        subj_id = subj_list[idx] if idx < len(subj_list) else idx
        n_e, n_l = len(e_seq)-1, len(l_seq)-1

        ll1_e = log_likelihood_first_order(e_seq, mat_early_1st)
        ll2_e = log_likelihood_second_order_scratch(e_seq, mat2_early)
        ll1_l = log_likelihood_first_order(l_seq, mat_late_1st)
        ll2_l = log_likelihood_second_order_scratch(l_seq, mat2_late)

        a1e = aic(ll1_e, K_FIRST_ORDER);  a2e = aic(ll2_e, K_SECOND_ORDER)
        a1l = aic(ll1_l, K_FIRST_ORDER);  a2l = aic(ll2_l, K_SECOND_ORDER)
        b1e = bic(ll1_e, K_FIRST_ORDER, n_e); b2e = bic(ll2_e, K_SECOND_ORDER, n_e)
        b1l = bic(ll1_l, K_FIRST_ORDER, n_l); b2l = bic(ll2_l, K_SECOND_ORDER, n_l)

        totals["a1e"]+=a1e; totals["a2e"]+=a2e; totals["b1e"]+=b1e; totals["b2e"]+=b2e
        totals["a1l"]+=a1l; totals["a2l"]+=a2l; totals["b1l"]+=b1l; totals["b2l"]+=b2l

        print(f"  {subj_id:<6} {'Early':<8} {a1e:>9.1f} {a2e:>9.1f} {b1e:>9.1f} {b2e:>9.1f} {'1st' if a2e>a1e else '2nd':>8}")
        print(f"  {'':<6} {'Late':<8} {a1l:>9.1f} {a2l:>9.1f} {b1l:>9.1f} {b2l:>9.1f} {'1st' if a2l>a1l else '2nd':>8}")

    print(f"\n  {group_name} TOTALS:")
    print(f"  {'':10} {'AIC 1st':>10} {'AIC 2nd':>10} {'BIC 1st':>10} {'BIC 2nd':>10} {'Verdict':>10}")
    print(f"  {'-'*55}")
    for window, a1, a2, b1, b2 in [
        ("Early", totals["a1e"], totals["a2e"], totals["b1e"], totals["b2e"]),
        ("Late",  totals["a1l"], totals["a2l"], totals["b1l"], totals["b2l"]),
    ]:
        print(f"  {window:<10} {a1:>10.1f} {a2:>10.1f} {b1:>10.1f} {b2:>10.1f} {'1st order' if a2>a1 else '2nd order':>10}")

: 

In [ ]:
#STATIONARY DISTRIBUTION VALIDATION
#Compares model-predicted stationary distribution to the actual observed stage proportions in the data.
#If they are close, the Markov model is a valid representation.
#Used the first-order model because the above analysis showed us that first-order worked better

def stationary_dist(trans_matrix):
    """Stationary distribution via left eigenvector of transition matrix."""
    evals, evecs = np.linalg.eig(trans_matrix.T)
    idx = np.argmin(np.abs(evals - 1))
    pi = np.real(evecs[:, idx])
    return pi / np.sum(pi)

def observed_proportions(sequences):
    """Actual proportion of time spent in each state across all sequences."""
    all_states = np.concatenate(sequences)
    n = len(all_states)
    return np.array([(all_states == s).sum() / n for s in range(3)])

STAGE_LABELS = ["W", "N", "R"]

print("\n" + "="*55)
print("STATIONARY DISTRIBUTION VALIDATION")
print("If predicted ≈ observed, the Markov model is valid.")
print("="*55)

for group_name, mat_early, mat_late, early_seqs, late_seqs in [
    ("Female", female_early, female_late, f_early_seqs, f_late_seqs),
    ("Male",   male_early,   male_late,   m_early_seqs, m_late_seqs),
]:
    print(f"\n{group_name}")
    print(f"  {'':22} {'W':>8} {'N':>8} {'R':>8}")
    print(f"  {'-'*46}")

    for window_label, mat, seqs in [
        ("Early (hrs 8-12)", mat_early, early_seqs),
        ("Late  (hrs 12-16)", mat_late,  late_seqs),
    ]:
        predicted = stationary_dist(mat)
        observed  = observed_proportions(seqs)
        error     = np.abs(predicted - observed)

        print(f"  {window_label}")
        print(f"  {'  Predicted':22} {predicted[0]:>8.3f} {predicted[1]:>8.3f} {predicted[2]:>8.3f}")
        print(f"  {'  Observed':22} {observed[0]:>8.3f} {observed[1]:>8.3f} {observed[2]:>8.3f}")
        print(f"  {'  Abs error':22} {error[0]:>8.3f} {error[1]:>8.3f} {error[2]:>8.3f}")
        print(f"  {'  Mean abs error':22} {error.mean():>8.3f}")
        print()

: 

In [ ]:
#generate sample sequence of states

def generate_sample_seq(list_states, init_state, trans_probs, num_steps):
    sample_states = np.zeros(num_steps, dtype=int)
    sample_states[0] = init_state
    for i in np.arange(num_steps-1):
            current = sample_states[i] #current state
            new = np.random.choice(list_states, p = trans_probs[current,:]) #next state
            sample_states[i+1] = new
    return sample_states


: 